# 🌊 SLCCI vs CMEMS — Full Comparative Analysis## Along-track altimetry (SLCCI) vs gridded geostrophic currents (CMEMS)This notebook compares **SLCCI** and **CMEMS** datasets across all analysis tabs:| # | Section | SLCCI | CMEMS ||---|---------|-------|-------|| 1 | Load Data | ✅ Cycles → DOT | ✅ Gate NetCDF || 2 | Slope Timeline | ✅ DOT slope | ❌ N/A || 3 | DOT Profile | ✅ DOT profile | ❌ N/A || 4 | Spatial Map | ✅ Pass track | ✅ Gate grid || 5 | Monthly Analysis | ✅ DOT climatology | ❌ N/A || 6 | Geostrophic Velocity | ✅ v_geo (uniform) | ✅ v⊥ (per-point, daily) || 7 | Volume Transport | ✅ VT monthly | ✅ VT daily || 8 | Salinity Profile | ✅ ISAS25 PSAL | ✅ ISAS PSAL (from NetCDF) || 9 | Freshwater Transport | ✅ FW monthly | ✅ FW daily (ISAS clim) || 10 | Salt Flux | ✅ SF monthly | ✅ SF daily (ISAS clim) || 11 | Temporal Aggregation | ✅ Climatology | ✅ Climatology |**SLCCI method**: DOT = corrected SSH − geoid → slope → v_geo = −(g/f) × ∂η/∂x (uniform)**CMEMS method**: ugos/vgos L4 daily → local per-point v⊥ projection (spatially resolved)

In [ ]:
# ════════════════════════════════════════════════════════════════
# 📚 CELL 1: IMPORTS
# ════════════════════════════════════════════════════════════════

import sys
import glob
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import geopandas as gpd
from pathlib import Path
from scipy import stats
from scipy.interpolate import RegularGridInterpolator
from shapely.geometry import LineString
import warnings
warnings.filterwarnings('ignore', 'Mean of empty slice')

# ── Add project root to path ──
PROJECT_ROOT = Path("__file__").resolve().parent.parent if "__file__" in dir() else Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

# ── src/slcci — pure I/O & DOT ──
from src.slcci.loader import load_cycles_serial
from src.slcci.geoid import interpolate_geoid, load_geoid
from src.slcci.dot import compute_dot, build_dataframe, build_dot_matrix, compute_slope_series
from src.slcci.spatial import gate_bounds, gate_profile_points, filter_near_gate
from src.slcci.binning import longitude_bin, mean_profile_pooled, monthly_climatology_profiles

# ── src/physics — geostrophy, transport, aggregation ──
from src.physics.constants import GRAVITY, DEPTH_CAP, S_REF, RHO, SVERDRUP, coriolis_parameter
from src.physics.geostrophy import geostrophic_velocity_from_slope
from src.physics.transport import volume_transport, freshwater_transport, salt_flux
from src.physics.aggregation import monthly_mean, monthly_climatology, annual_mean, rolling_mean

# ── CMEMS utils (from CMEMS repo) ──
CMEMS_ANALYSIS_DIR = "/Users/nicolocaron/Documents/GitHub/CMEMS/analysis"
sys.path.insert(0, CMEMS_ANALYSIS_DIR)
import importlib
import utils as cmems_utils
importlib.reload(cmems_utils)
from utils import (
    load_gate,
    perpendicular_velocity as cmems_perpendicular_velocity,
    perpendicular_velocity_uncertainty as cmems_perpendicular_velocity_uncertainty,
    volume_transport as cmems_volume_transport,
    volume_transport_uncertainty as cmems_volume_transport_uncertainty,
    freshwater_transport as cmems_freshwater_transport,
    freshwater_transport_uncertainty as cmems_freshwater_transport_uncertainty,
    salt_flux as cmems_salt_flux,
    salt_flux_uncertainty as cmems_salt_flux_uncertainty,
    monthly_along_gate_profile,
    SVERDRUP as CMEMS_SVERDRUP,
)

# ── Matplotlib defaults ──
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "grid.linestyle": ":",
})

print("✅ All imports OK (SLCCI + CMEMS)")

In [ ]:
# ════════════════════════════════════════════════════════════════
# ⚙️ CELL 2: CONFIGURATION
# ════════════════════════════════════════════════════════════════

# ── 1. STRAIT / PASS ──
SELECTED_STRAIT = "davis"          # "davis", "fram", "bering"
PASS_NUMBER     = 248

# ── 2. DATA PATHS — SLCCI ──
BASE_DIR   = Path("/Users/nicolocaron/Desktop/ARCFRESH/DATA SOURCES/J2")
GEOID_PATH = "/Users/nicolocaron/Desktop/ARCFRESH/DATA SOURCES/TUM_ogmoc.nc"
GEBCO_PATH = "/Users/nicolocaron/Desktop/ARCFRESH/DATA SOURCES/GEBCO_06_Feb_2026_c91df93f54b8/gebco_2025_n90.0_s55.0_w0.0_e360.0.nc"
PSAL_DIR   = Path("/Users/nicolocaron/Desktop/ARCFRESH/DATA SOURCES/ISAS25_PSAL")
GATES_FOLDER = str(PROJECT_ROOT / "gates")

# ── 3. DATA PATHS — CMEMS ──
CMEMS_NC_DIR = Path("/Users/nicolocaron/Desktop/ARCFRESH/DATA SOURCES/ALL STRAITS DATASET netCDF")

# Strait → CMEMS NetCDF file map
STRAIT_CMEMS_MAP = {
    "davis": "arcfresh_davis_strait_2002-2023.nc",
    "fram": "arcfresh_fram_strait_2002-2023.nc",
    "bering": "arcfresh_bering_strait_2002-2023.nc",
}

# ── 4. PROCESSING ──
USE_FLAG      = True
LON_BIN_SIZE  = 0.01      # fine binning for slope/matrix
PROFILE_BIN   = 0.10      # coarser binning for profiles
DEPTH_CAP_M   = 250.0     # max integration depth [m]
S_REF_PSU     = 34.8      # reference salinity [PSU]
RHO_CMEMS     = 1024.0    # CMEMS uses 1024, SLCCI uses 1025

# ── 5. OUTPUT ──
SAVE_OUTPUTS = True
OUTPUT_DIR   = Path("/Users/nicolocaron/Desktop/ARCFRESH/plot slccii/FULL_ANALYSIS_COMPARISON")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Satellite auto-detect ──
mission = BASE_DIR.name.upper()
if mission == "J2":
    cycles = list(range(1, 282))
    satellite_name = "J2"
elif mission == "J1":
    cycles = list(range(1, 538))
    satellite_name = "J1"
else:
    cycles = list(range(1, 282))
    satellite_name = mission

# ── Strait → file pattern map ──
STRAIT_PATTERNS = {
    "davis": "*davis_strait*.shp",
    "fram": "*fram*.shp",
    "bering": "*bering*.shp",
}

# ── Strait → salinity file map ──
STRAIT_PSAL_MAP = {
    "davis": "davis_strait_CLIM_ISAS25_PSAL.nc",
    "fram": "fram_strait_S3_pass_481_CLIM_ISAS25_PSAL.nc",
    "bering": "bering_strait_TPJ_pass_076_CLIM_ISAS25_PSAL.nc",
}

print("\n" + "=" * 70)
print("⚙️  CONFIGURATION — SLCCI vs CMEMS COMPARISON")
print("=" * 70)
print(f"🌊 Strait:       {SELECTED_STRAIT.upper()}")
print(f"🛰️  Satellite:    {satellite_name} ({len(cycles)} cycles)")
print(f"🔢 Pass:         {PASS_NUMBER}")
print(f"📂 SLCCI dir:    {BASE_DIR}")
print(f"📂 CMEMS file:   {STRAIT_CMEMS_MAP.get(SELECTED_STRAIT, 'N/A')}")
print(f"📂 Gates:        {GATES_FOLDER}")
print(f"🌍 GEBCO:        {Path(GEBCO_PATH).name}")
print(f"🧂 Salinity:     {STRAIT_PSAL_MAP.get(SELECTED_STRAIT, 'N/A')}")
print(f"📊 Bin sizes:    slope={LON_BIN_SIZE}°, profile={PROFILE_BIN}°")
print(f"🏊 Depth cap:    {DEPTH_CAP_M} m")
print(f"💾 Output:       {OUTPUT_DIR}")
print("=" * 70)

In [ ]:
# ════════════════════════════════════════════════════════════════
# 📡 CELL 3: LOAD DATA — SLCCI (Cycles → DOT) + CMEMS (Gate NetCDF)
# ════════════════════════════════════════════════════════════════

# ══════════════════════════════════════════════════════════════
# 3A. SLCCI DATA
# ══════════════════════════════════════════════════════════════
print("━" * 70)
print("📡 LOADING SLCCI DATA")
print("━" * 70)

# ── 3a.1 Load gate shapefile ──
pattern = str(Path(GATES_FOLDER) / STRAIT_PATTERNS[SELECTED_STRAIT.lower()])
gate_files = [f for f in glob.glob(pattern) if "_east" not in f and "_west" not in f]
if not gate_files:
    raise FileNotFoundError(f"No gate file found for '{SELECTED_STRAIT}' with pattern {pattern}")
GATE_PATH = gate_files[0]

strait_name = Path(GATE_PATH).stem.replace("_", " ").title()
gate = gpd.read_file(GATE_PATH).to_crs("EPSG:4326")
gate_geom_bounds = gate.total_bounds  # (minx, miny, maxx, maxy)

lat_min, lat_max, lon_min_360, lon_max_360 = gate_bounds(GATE_PATH, lat_buffer=2.0, lon_buffer=5.0)
print(f"📍 Gate: {strait_name}")
print(f"   Bounds: lon [{gate_geom_bounds[0]:.2f}, {gate_geom_bounds[2]:.2f}], "
      f"lat [{gate_geom_bounds[1]:.2f}, {gate_geom_bounds[3]:.2f}]")

# ── 3a.2 Load satellite cycles ──
print(f"\n🔄 Loading {satellite_name} cycles for Pass {PASS_NUMBER}...")
ds_pass = load_cycles_serial(
    base_dir=str(BASE_DIR),
    cycles=cycles,
    pass_number=PASS_NUMBER,
    lat_bounds=(lat_min, lat_max),
    lon_bounds=(lon_min_360, lon_max_360),
    use_flag=USE_FLAG,
    satellite=satellite_name,
)
if ds_pass is None or ds_pass.sizes.get("time", 0) == 0:
    raise ValueError(f"No data found for Pass {PASS_NUMBER}")
print(f"   Raw observations: {ds_pass.sizes['time']:,}")

# ── 3a.3 Interpolate geoid ──
geoid_at_obs = interpolate_geoid(
    geoid_path=GEOID_PATH,
    target_lats=ds_pass["latitude"].values,
    target_lons=ds_pass["longitude"].values,
)

# ── 3a.4 Build DOT DataFrame ──
j2_data = build_dataframe(ds_pass, geoid_at_obs, PASS_NUMBER)
if j2_data is None or j2_data.empty:
    raise ValueError(f"No valid data after DOT computation for Pass {PASS_NUMBER}")

n_cycles_loaded = j2_data["cycle"].nunique()
mean_lat = j2_data["lat"].mean()

print(f"\n✅ SLCCI Data ready!")
print(f"   Observations: {len(j2_data):,}")
print(f"   Cycles: {n_cycles_loaded}")
print(f"   LON: [{j2_data['lon'].min():.3f}, {j2_data['lon'].max():.3f}]")
print(f"   LAT: [{j2_data['lat'].min():.3f}, {j2_data['lat'].max():.3f}]  (mean: {mean_lat:.3f}°)")
print(f"   Time: {j2_data['time'].min()} → {j2_data['time'].max()}")

# ══════════════════════════════════════════════════════════════
# 3B. CMEMS DATA
# ══════════════════════════════════════════════════════════════
print("\n" + "━" * 70)
print("🌐 LOADING CMEMS DATA")
print("━" * 70)

cmems_nc_file = CMEMS_NC_DIR / STRAIT_CMEMS_MAP[SELECTED_STRAIT]
ds_cmems = load_gate(cmems_nc_file)

cmems_gate_name = ds_cmems.attrs.get('gate_display_name', SELECTED_STRAIT)
cmems_lon = ds_cmems['longitude'].values
cmems_lat = ds_cmems['latitude'].values
cmems_x_km = ds_cmems['x_km'].values
cmems_time = pd.to_datetime(ds_cmems['time'].values)

# Check salinity availability (ISAS climatology only)
cmems_has_isas = ('psal_isas_surface' in ds_cmems) and bool(np.isfinite(ds_cmems['psal_isas_surface'].values).any())
cmems_sal_var = 'psal_isas_surface'
cmems_sal_label = 'ISAS PSAL climatology'

print(f"📍 Gate: {cmems_gate_name}")
print(f"   Points: {ds_cmems.sizes['point']}  |  Days: {ds_cmems.sizes['time']}")
print(f"   Period: {str(cmems_time[0].date())} → {str(cmems_time[-1].date())}")
print(f"   Length: {cmems_x_km[-1]:.1f} km")
print(f"   Lon: {cmems_lon[0]:.2f}° → {cmems_lon[-1]:.2f}°")
print(f"   Lat: {cmems_lat[0]:.2f}° → {cmems_lat[-1]:.2f}°")
print(f"   ISAS PSAL: {'AVAILABLE' if cmems_has_isas else 'NOT AVAILABLE'}")
print(f"   Depth cap: {DEPTH_CAP_M} m")

# ── Pre-compute CMEMS v_perp (used in multiple cells) ──
cmems_v_perp = cmems_perpendicular_velocity(ds_cmems)  # (point, time)
cmems_vp_mean_ts = np.nanmean(cmems_v_perp, axis=0)     # gate-mean daily

print(f"\n✅ CMEMS Data ready!")
print(f"   v⊥ matrix: {cmems_v_perp.shape} (point × time)")
print(f"   v⊥ gate-mean range: [{np.nanmin(cmems_vp_mean_ts):.4f}, {np.nanmax(cmems_vp_mean_ts):.4f}] m/s")

In [ ]:
# ════════════════════════════════════════════════════════════════
# 📈 CELL 4: SLOPE TIMELINE  (SLCCI only — CMEMS has no DOT slope)
# ════════════════════════════════════════════════════════════════

dot_matrix, time_periods, lon_centers, x_km = build_dot_matrix(j2_data, lon_bin_size=LON_BIN_SIZE)
slope_series = compute_slope_series(dot_matrix, x_km)
time_array = np.array([pd.Timestamp(str(p)) for p in time_periods])

print(f"DOT matrix: {dot_matrix.shape[0]} lon bins × {dot_matrix.shape[1]} months")
print(f"Slope range: [{np.nanmin(slope_series):.4f}, {np.nanmax(slope_series):.4f}] m/100km")

# ── Plot ──
fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(time_array, slope_series, "-o", markersize=3, linewidth=1.2, color="steelblue")
ax.axhline(0, color="k", linewidth=0.8)
ax.set_ylabel("DOT Slope (m / 100 km)", fontsize=12)
ax.set_title(f"📈 Slope Timeline — {strait_name} — {satellite_name} Pass {PASS_NUMBER}\n"
             f"{time_array[0].strftime('%Y-%m')} to {time_array[-1].strftime('%Y-%m')}  [SLCCI only]",
             fontsize=14, fontweight="bold")
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.xticks(rotation=45, ha="right")

mean_s = np.nanmean(slope_series)
std_s = np.nanstd(slope_series)
ax.axhline(mean_s, color="red", linewidth=1, linestyle="--", alpha=0.7, label=f"Mean: {mean_s:.4f}")
ax.fill_between(time_array, mean_s - std_s, mean_s + std_s, color="red", alpha=0.08, label=f"±1σ: {std_s:.4f}")
ax.legend(fontsize=10)

plt.tight_layout()
if SAVE_OUTPUTS:
    fname = f"01_slope_timeline_{satellite_name}_pass{PASS_NUMBER}_{SELECTED_STRAIT}.png"
    plt.savefig(OUTPUT_DIR / fname, dpi=300, bbox_inches="tight")
    print(f"💾 Saved: {fname}")
plt.show()
print("✅ Slope timeline complete")

In [ ]:
# ════════════════════════════════════════════════════════════════
# 📊 CELL 5: DOT PROFILE  (SLCCI only)
# ════════════════════════════════════════════════════════════════

profile_mean, lon_centers_p, x_km_p = mean_profile_pooled(j2_data, lon_bin_size=PROFILE_BIN)
valid_p = np.isfinite(profile_mean)

slope_prof_100km = np.nan
if np.sum(valid_p) >= 2:
    slope_prof, intercept_prof = np.polyfit(x_km_p[valid_p], profile_mean[valid_p], 1)
    fit_line = slope_prof * x_km_p + intercept_prof
    slope_prof_100km = slope_prof * 100.0

fig, ax = plt.subplots(figsize=(14, 5))
if np.any(valid_p):
    ax.plot(x_km_p[valid_p], profile_mean[valid_p], "o-", color="darkblue",
            markersize=5, linewidth=2, label="Mean DOT")
    if np.sum(valid_p) >= 2:
        ax.plot(x_km_p, fit_line, "--", color="red", linewidth=1.5,
                label=f"Linear fit: {slope_prof_100km:.4f} m/100km")

ax.set_xlabel("Distance along gate (km)", fontsize=12)
ax.set_ylabel("DOT (m)", fontsize=12)
ax.set_title(f"📊 Mean DOT Profile — {strait_name} — {satellite_name} Pass {PASS_NUMBER}\n"
             f"Pooled across {len(time_periods)} months, {PROFILE_BIN}° bins  [SLCCI only]",
             fontsize=14, fontweight="bold")

if np.any(valid_p):
    y_range = np.nanmax(profile_mean[valid_p]) - np.nanmin(profile_mean[valid_p])
    y_top = np.nanmax(profile_mean[valid_p]) - 0.05 * y_range if y_range > 0 else np.nanmean(profile_mean[valid_p])
    ax.text(x_km_p.min(), y_top, "WEST", fontsize=11, fontweight="bold", ha="left", va="top")
    ax.text(x_km_p.max(), y_top, "EAST", fontsize=11, fontweight="bold", ha="right", va="top")

ax.legend(fontsize=10)
plt.tight_layout()
if SAVE_OUTPUTS:
    fname = f"02_dot_profile_{satellite_name}_pass{PASS_NUMBER}_{SELECTED_STRAIT}.png"
    plt.savefig(OUTPUT_DIR / fname, dpi=300, bbox_inches="tight")
    print(f"💾 Saved: {fname}")
plt.show()
print("✅ DOT profile complete")

In [ ]:
# ════════════════════════════════════════════════════════════════
# 🗺️ CELL 6: SPATIAL MAP — SLCCI pass + CMEMS gate geometry
# ════════════════════════════════════════════════════════════════

mean_dot_by_loc = j2_data.groupby(["lon", "lat"]).agg({"dot": "mean"}).reset_index()

proj = ccrs.PlateCarree()
fig, ax = plt.subplots(figsize=(14, 10), subplot_kw=dict(projection=proj))

# Map extent — use wider bounds to show both geometries
all_lons = np.concatenate([mean_dot_by_loc["lon"].values, cmems_lon])
all_lats = np.concatenate([mean_dot_by_loc["lat"].values, cmems_lat])
LON_BUF, LAT_BUF = 3.0, 2.0
ax.set_extent([
    all_lons.min() - LON_BUF, all_lons.max() + LON_BUF,
    all_lats.min() - LAT_BUF, all_lats.max() + LAT_BUF,
], crs=proj)

ax.coastlines(linewidth=1.0)
ax.add_feature(cfeature.LAND, facecolor="beige", alpha=0.7)
ax.add_feature(cfeature.OCEAN, facecolor="lightblue", alpha=0.3)
gl = ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False,
                   linewidth=0.8, alpha=0.6, linestyle="--", color="gray")
gl.top_labels = False
gl.right_labels = False

# 1. Gate shapefile (the "real" gate line)
gate.plot(ax=ax, color="red", linewidth=3, transform=proj, zorder=10, label="Gate (shapefile)")

# 2. CMEMS gate points (92 points on the grid)
ax.plot(cmems_lon, cmems_lat, "s-", color="limegreen", markersize=4, linewidth=2,
        transform=proj, zorder=8, alpha=0.8, label=f"CMEMS grid ({len(cmems_lon)} pts)")

# 3. SLCCI pass track (DOT scatter)
sc = ax.scatter(
    mean_dot_by_loc["lon"], mean_dot_by_loc["lat"],
    c=mean_dot_by_loc["dot"], s=15, alpha=0.7,
    cmap="viridis", transform=proj, zorder=5,
    edgecolors="white", linewidths=0.2,
)
plt.colorbar(sc, ax=ax, label="DOT (m) [SLCCI]", shrink=0.8, pad=0.02)

ax.legend(loc="upper left", fontsize=11, framealpha=0.95)
ax.set_title(
    f"🗺️ Spatial Map — {strait_name}\n"
    f"Red: gate shapefile | Green: CMEMS grid | Scatter: SLCCI Pass {PASS_NUMBER}",
    fontsize=14, fontweight="bold", pad=15,
)

plt.tight_layout()
if SAVE_OUTPUTS:
    fname = f"03_spatial_map_{satellite_name}_pass{PASS_NUMBER}_{SELECTED_STRAIT}.png"
    plt.savefig(OUTPUT_DIR / fname, dpi=300, bbox_inches="tight")
    print(f"💾 Saved: {fname}")
plt.show()
print("✅ Spatial map complete")

In [ ]:
# ════════════════════════════════════════════════════════════════
# 📅 CELL 7: MONTHLY ANALYSIS — 12 subplots  (SLCCI DOT only)
# ════════════════════════════════════════════════════════════════

monthly_profiles, lon_c_monthly, x_km_monthly = monthly_climatology_profiles(
    j2_data, lon_bin_size=PROFILE_BIN
)

MONTH_NAMES = {
    1: "January", 2: "February", 3: "March", 4: "April",
    5: "May", 6: "June", 7: "July", 8: "August",
    9: "September", 10: "October", 11: "November", 12: "December",
}

all_vals = np.concatenate([p[np.isfinite(p)] for p in monthly_profiles.values() if np.any(np.isfinite(p))])
if len(all_vals) > 0:
    dot_margin = 0.05 * (all_vals.max() - all_vals.min())
    ylim_global = (all_vals.min() - dot_margin, all_vals.max() + dot_margin)
else:
    ylim_global = None

fig, axes = plt.subplots(3, 4, figsize=(28, 18))
axes = axes.flatten()
monthly_results = []

for month_num in range(1, 13):
    ax = axes[month_num - 1]
    prof = monthly_profiles[month_num]
    valid = np.isfinite(prof)

    if not np.any(valid):
        ax.text(0.5, 0.5, f"{MONTH_NAMES[month_num]}\n\nNo data",
                ha="center", va="center", fontsize=13, transform=ax.transAxes)
        ax.set_xticks([]); ax.set_yticks([])
        continue

    ax.scatter(x_km_monthly[valid], prof[valid], s=40, alpha=0.75,
               color="steelblue", edgecolor="k", linewidth=0.3)

    if np.sum(valid) >= 2:
        slope_m, intercept_m = np.polyfit(x_km_monthly[valid], prof[valid], 1)
        fit_y = slope_m * x_km_monthly[valid] + intercept_m
        ax.plot(x_km_monthly[valid], fit_y, "--", linewidth=2, color="darkred")
        ss_res = np.sum((prof[valid] - fit_y) ** 2)
        ss_tot = np.sum((prof[valid] - np.mean(prof[valid])) ** 2)
        r2 = 1 - (ss_res / ss_tot) if ss_tot > 0 else np.nan
        slope_100km = slope_m * 100.0
        monthly_results.append({"month": month_num, "month_name": MONTH_NAMES[month_num],
                                 "slope_m_100km": slope_100km, "r_squared": r2, "n_bins": int(np.sum(valid))})
        ax.set_title(f"{MONTH_NAMES[month_num]}\nslope={slope_100km:.4f} m/100km | R²={r2:.3f}",
                     fontsize=11, fontweight="bold")
    else:
        ax.set_title(f"{MONTH_NAMES[month_num]}", fontsize=11, fontweight="bold")

    ax.set_xlabel("Distance (km)", fontsize=10)
    ax.set_ylabel("DOT (m)", fontsize=10)
    if ylim_global:
        ax.set_ylim(ylim_global)

fig.suptitle(
    f"📅 Monthly DOT Climatology — {strait_name} — {satellite_name} Pass {PASS_NUMBER}  [SLCCI only]\n"
    f"Binned at {PROFILE_BIN}° resolution with linear regression",
    fontsize=17, fontweight="bold", y=0.995)
plt.tight_layout(rect=[0, 0, 1, 0.97])
if SAVE_OUTPUTS:
    fname = f"04_monthly_analysis_{satellite_name}_pass{PASS_NUMBER}_{SELECTED_STRAIT}.png"
    plt.savefig(OUTPUT_DIR / fname, dpi=300, bbox_inches="tight")
    print(f"💾 Saved: {fname}")
plt.show()
print("✅ Monthly analysis complete")

In [ ]:
# ════════════════════════════════════════════════════════════════
# 🌀 CELL 8: GEOSTROPHIC VELOCITY — SLCCI vs CMEMS
# ════════════════════════════════════════════════════════════════
# SLCCI: v_geo = -(g/f) × ∂η/∂x  (uniform, monthly)
# CMEMS: v⊥ = ugos·n_x + vgos·n_y  (per-point, daily → gate-mean)

v_geo = geostrophic_velocity_from_slope(slope_series, mean_lat)

f_param = coriolis_parameter(mean_lat)
print(f"Mean latitude: {mean_lat:.3f}°")
print(f"Coriolis parameter: f = {f_param:.6e} rad/s")
print(f"SLCCI v_geo: mean={np.nanmean(v_geo):.4f} ± {np.nanstd(v_geo):.4f} m/s")
print(f"CMEMS v⊥:   mean={np.nanmean(cmems_vp_mean_ts):.4f} ± {np.nanstd(cmems_vp_mean_ts):.4f} m/s")

# ── Plot: comparison ──
fig, ax = plt.subplots(figsize=(16, 6))

# CMEMS daily (thin line, background)
ax.plot(cmems_time, cmems_vp_mean_ts, linewidth=0.5, alpha=0.4, color="coral",
        label=f"CMEMS v⊥ daily (mean={np.nanmean(cmems_vp_mean_ts):.4f} m/s)")

# SLCCI monthly (markers)
ax.plot(time_array, v_geo, "-o", markersize=4, linewidth=1.5, color="darkgreen",
        label=f"SLCCI v_geo monthly (mean={np.nanmean(v_geo):.4f} m/s)", zorder=5)

ax.axhline(0, color="k", linewidth=0.8)
ax.axhline(np.nanmean(v_geo), color="darkgreen", linestyle="--", linewidth=1, alpha=0.5)
ax.axhline(np.nanmean(cmems_vp_mean_ts), color="coral", linestyle="--", linewidth=1, alpha=0.5)

ax.set_ylabel("Velocity (m/s)", fontsize=12)
ax.set_xlabel("Date", fontsize=12)
ax.set_title(
    f"🌀 Geostrophic Velocity — {strait_name} — SLCCI vs CMEMS\n"
    f"SLCCI: DOT slope method (uniform) | CMEMS: per-point v⊥ projection (gate-mean)",
    fontsize=14, fontweight="bold")
ax.legend(fontsize=10, loc="best")
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.xticks(rotation=45, ha="right")
plt.tight_layout()

if SAVE_OUTPUTS:
    fname = f"05_velocity_comparison_{satellite_name}_pass{PASS_NUMBER}_{SELECTED_STRAIT}.png"
    plt.savefig(OUTPUT_DIR / fname, dpi=300, bbox_inches="tight")
    print(f"💾 Saved: {fname}")
plt.show()
print("✅ Geostrophic velocity comparison complete")

In [ ]:
# ════════════════════════════════════════════════════════════════
# 🏔️ CELL 9: LOAD BATHYMETRY (GEBCO) + GATE GEOMETRY
# ════════════════════════════════════════════════════════════════

print("🔄 Loading GEBCO bathymetry...")
ds_gebco = xr.open_dataset(GEBCO_PATH)

gate_lon, gate_lat, gate_x_km = gate_profile_points(GATE_PATH, n_points=100)
print(f"   Gate sampled: {len(gate_lon)} points, length: {gate_x_km[-1]:.1f} km")

gebco_lat = ds_gebco["lat"].values
gebco_lon = ds_gebco["lon"].values
gebco_elev = ds_gebco["elevation"].values

interp_gebco = RegularGridInterpolator(
    (gebco_lat, gebco_lon), gebco_elev.astype(float),
    method="linear", bounds_error=False, fill_value=np.nan)

gate_lon_360 = gate_lon % 360
gate_elevation = interp_gebco(np.column_stack([gate_lat, gate_lon_360]))
gate_depth = np.abs(gate_elevation)
gate_depth = np.where(gate_elevation > 0, 0, gate_depth)
gate_depth_capped = np.minimum(gate_depth, DEPTH_CAP_M)
dx_m = np.gradient(gate_x_km * 1000.0)

ds_gebco.close()

print(f"   Depth range: {gate_depth.min():.0f} – {gate_depth.max():.0f} m")
print(f"   Depth capped at {DEPTH_CAP_M:.0f} m")

# ── Plot ──
fig, ax = plt.subplots(figsize=(14, 4))
ax.fill_between(gate_x_km, 0, -gate_depth, color="saddlebrown", alpha=0.5, label="Bathymetry (SLCCI gate)")

# Also show CMEMS bathymetry from NetCDF
cmems_depth_vals = ds_cmems['depth'].values
ax.plot(cmems_x_km, -cmems_depth_vals, color="limegreen", linewidth=2, alpha=0.7,
        label=f"Bathymetry (CMEMS, {len(cmems_depth_vals)} pts)")

ax.axhline(-DEPTH_CAP_M, color="red", linestyle="--", linewidth=1.5, label=f"Depth cap ({DEPTH_CAP_M:.0f} m)")
ax.set_xlabel("Distance along gate (km)", fontsize=12)
ax.set_ylabel("Elevation (m)", fontsize=12)
ax.set_title(f"🏔️ GEBCO Bathymetry along {strait_name} — SLCCI vs CMEMS gate", fontsize=14, fontweight="bold")
ax.legend(fontsize=10)
ax.set_ylim(-min(max(gate_depth.max(), cmems_depth_vals.max()) * 1.1, 1000), 50)
plt.tight_layout()
if SAVE_OUTPUTS:
    fname = f"06_bathymetry_{SELECTED_STRAIT}.png"
    plt.savefig(OUTPUT_DIR / fname, dpi=300, bbox_inches="tight")
    print(f"💾 Saved: {fname}")
plt.show()
print("✅ Bathymetry loaded")

In [ ]:
# ════════════════════════════════════════════════════════════════
# 🌊 CELL 10: VOLUME TRANSPORT — SLCCI vs CMEMS
# ════════════════════════════════════════════════════════════════

# ── SLCCI VT ──
n_gate_pts = len(gate_lon)
n_time = len(v_geo)
v_perp_matrix = np.tile(v_geo, (n_gate_pts, 1))  # uniform v_geo

vt_sv = volume_transport(
    v_perp=v_perp_matrix, depth=gate_depth, dx=dx_m, depth_cap=DEPTH_CAP_M)

# ── CMEMS VT (daily, from utils) ──
cmems_vt_sv, cmems_vt_time = cmems_volume_transport(ds_cmems, depth_cap=DEPTH_CAP_M)

print(f"SLCCI VT:  mean={np.nanmean(vt_sv):.3f} ± {np.nanstd(vt_sv):.3f} Sv  (monthly, {len(vt_sv)} pts)")
print(f"CMEMS VT:  mean={np.nanmean(cmems_vt_sv):.3f} ± {np.nanstd(cmems_vt_sv):.3f} Sv  (daily, {len(cmems_vt_sv)} pts)")

# ── Plot ──
fig, ax = plt.subplots(figsize=(16, 6))

# CMEMS daily
ax.plot(cmems_vt_time, cmems_vt_sv, linewidth=0.5, alpha=0.35, color="coral",
        label=f"CMEMS daily (mean={np.nanmean(cmems_vt_sv):.3f} Sv)")

# SLCCI monthly
ax.plot(time_array, vt_sv, "-o", markersize=4, linewidth=1.5, color="teal",
        label=f"SLCCI monthly (mean={np.nanmean(vt_sv):.3f} Sv)", zorder=5)

ax.axhline(0, color="k", linewidth=0.8)
ax.axhline(np.nanmean(vt_sv), color="teal", linestyle="--", linewidth=1, alpha=0.5)
ax.axhline(np.nanmean(cmems_vt_sv), color="coral", linestyle="--", linewidth=1, alpha=0.5)

ax.set_ylabel("Volume Transport (Sv)", fontsize=12)
ax.set_title(
    f"🌊 Volume Transport — {strait_name} — SLCCI vs CMEMS\n"
    f"Depth cap: {DEPTH_CAP_M:.0f} m | Positive = into Arctic",
    fontsize=14, fontweight="bold")
ax.legend(fontsize=10)
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.xticks(rotation=45, ha="right")
plt.tight_layout()

if SAVE_OUTPUTS:
    fname = f"07_volume_transport_comparison_{satellite_name}_pass{PASS_NUMBER}_{SELECTED_STRAIT}.png"
    plt.savefig(OUTPUT_DIR / fname, dpi=300, bbox_inches="tight")
    print(f"💾 Saved: {fname}")
plt.show()
print("✅ Volume transport comparison complete")

In [ ]:
# ════════════════════════════════════════════════════════════════
# 🧂 CELL 11: SALINITY PROFILE — SLCCI (ISAS25) vs CMEMS (ISAS clim)
# ════════════════════════════════════════════════════════════════

# ── SLCCI: ISAS25 PSAL climatology ──
psal_file = PSAL_DIR / STRAIT_PSAL_MAP[SELECTED_STRAIT]
print(f"🔄 Loading SLCCI salinity: {psal_file.name}")

ds_psal = xr.open_dataset(psal_file)
psal_lat = ds_psal["latitude"].values
psal_lon = ds_psal["longitude"].values
psal_depth = ds_psal["depth"].values
psal_data = ds_psal["PSAL"].values  # (time=12, z, nb_prof)

# ── CMEMS: ISAS PSAL surface from NetCDF ──
cmems_isas_sal = ds_cmems['psal_isas_surface'].values if cmems_has_isas else None  # (point, time)

# ── Plot: 1×2 — depth profiles + monthly SSS comparison ──
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))

# Left: annual mean salinity vs depth (SLCCI ISAS25 only — CMEMS has surface only)
annual_mean_psal = np.nanmean(psal_data, axis=0)  # (z, nb_prof)
for ip in range(annual_mean_psal.shape[1]):
    profile = annual_mean_psal[:, ip]
    valid_z = np.isfinite(profile)
    if np.any(valid_z):
        ax1.plot(profile[valid_z], psal_depth[valid_z], alpha=0.4, linewidth=0.8, color="steelblue")

overall_profile = np.nanmean(annual_mean_psal, axis=1)
valid_overall = np.isfinite(overall_profile)
ax1.plot(overall_profile[valid_overall], psal_depth[valid_overall],
         linewidth=3, color="darkblue", label="ISAS25 mean profile")

ax1.axhline(DEPTH_CAP_M, color="red", linestyle="--", linewidth=1.5, label=f"Depth cap ({DEPTH_CAP_M:.0f} m)")
ax1.axvline(S_REF_PSU, color="orange", linestyle=":", linewidth=1.5, label=f"S_ref = {S_REF_PSU} PSU")
ax1.invert_yaxis()
ax1.set_xlabel("Salinity (PSU)", fontsize=12)
ax1.set_ylabel("Depth (m)", fontsize=12)
ax1.set_title("Annual Mean Salinity Profiles (ISAS25)", fontsize=13, fontweight="bold")
ax1.legend(fontsize=10)
ax1.set_ylim(min(psal_depth.max(), 1000), 0)

# Right: monthly SSS along gate — SLCCI ISAS25 + CMEMS ISAS
month_labels = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
colors = plt.cm.twilight_shifted(np.linspace(0, 1, 12))

# SLCCI ISAS25 SSS (surface, z=0)
for im in range(12):
    sss_month = psal_data[im, 0, :]
    ax2.plot(psal_lon, sss_month, "o-", markersize=4, linewidth=1.2,
             color=colors[im], label=f"ISAS25 {month_labels[im]}" if im == 0 else None, alpha=0.7)

# CMEMS ISAS surface climatology (compute monthly mean from daily repeated clim)
if cmems_isas_sal is not None:
    cmems_months_idx = cmems_time.month
    for im in range(12):
        mask_m = cmems_months_idx == (im + 1)
        if mask_m.sum() > 0:
            cmems_sss_m = np.nanmean(cmems_isas_sal[:, mask_m], axis=1)  # (point,)
            ax2.plot(cmems_lon, cmems_sss_m, "x--", markersize=5, linewidth=1,
                     color=colors[im], alpha=0.5,
                     label=f"CMEMS ISAS {month_labels[im]}" if im == 0 else None)

ax2.set_xlabel("Longitude (°)", fontsize=12)
ax2.set_ylabel("SSS (PSU)", fontsize=12)
ax2.set_title("Monthly SSS: ISAS25 (●) vs CMEMS ISAS (×)", fontsize=13, fontweight="bold")
ax2.legend(fontsize=9, ncol=2, loc="best")

fig.suptitle(f"🧂 Salinity Profile — {strait_name} — SLCCI vs CMEMS",
             fontsize=15, fontweight="bold", y=1.01)
plt.tight_layout()

if SAVE_OUTPUTS:
    fname = f"08_salinity_comparison_{SELECTED_STRAIT}.png"
    plt.savefig(OUTPUT_DIR / fname, dpi=300, bbox_inches="tight")
    print(f"💾 Saved: {fname}")
plt.show()
ds_psal.close()
print("✅ Salinity comparison complete")

In [ ]:
# ════════════════════════════════════════════════════════════════
# 💧 CELL 12: FRESHWATER TRANSPORT — SLCCI vs CMEMS
# ════════════════════════════════════════════════════════════════

print("🔄 Computing freshwater transport...")

# ── SLCCI: Map ISAS25 SSS to gate points ──
ds_psal = xr.open_dataset(PSAL_DIR / STRAIT_PSAL_MAP[SELECTED_STRAIT])
psal_lon_isas = ds_psal["longitude"].values
psal_sss_monthly = ds_psal["PSAL"].values[:, 0, :]  # (12, nb_prof)

sss_at_gate = np.full((12, n_gate_pts), np.nan)
for im in range(12):
    sss_month = psal_sss_monthly[im, :]
    valid_isas = np.isfinite(sss_month)
    if np.sum(valid_isas) < 2:
        continue
    for ig in range(n_gate_pts):
        dist = np.abs(psal_lon_isas[valid_isas] - gate_lon[ig])
        idx_nearest = np.argmin(dist)
        sss_at_gate[im, ig] = sss_month[valid_isas][idx_nearest]

time_months = np.array([pd.Timestamp(str(p)).month for p in time_periods])
sss_matrix = np.full((n_gate_pts, n_time), np.nan)
for it in range(n_time):
    month_idx = time_months[it] - 1
    sss_matrix[:, it] = sss_at_gate[month_idx, :]

fw_m3s = freshwater_transport(
    v_perp=v_perp_matrix, sss=sss_matrix, depth=gate_depth,
    dx=dx_m, depth_cap=DEPTH_CAP_M, s_ref=S_REF_PSU)
fw_msv = fw_m3s / 1e3

# ── CMEMS FW (daily, ISAS climatology) ──
if cmems_has_isas:
    cmems_fw_m3s, cmems_fw_time = cmems_freshwater_transport(
        ds_cmems, depth_cap=DEPTH_CAP_M, s_ref=S_REF_PSU, sal_var='psal_isas_surface')
    cmems_fw_msv = cmems_fw_m3s / 1e3
else:
    cmems_fw_msv = np.array([])
    cmems_fw_time = pd.DatetimeIndex([])

print(f"SLCCI FW: mean={np.nanmean(fw_msv):.1f} ± {np.nanstd(fw_msv):.1f} mSv")
if len(cmems_fw_msv) > 0:
    print(f"CMEMS FW: mean={np.nanmean(cmems_fw_msv):.1f} ± {np.nanstd(cmems_fw_msv):.1f} mSv")

# ── Plot ──
fig, ax = plt.subplots(figsize=(16, 6))

if len(cmems_fw_msv) > 0:
    ax.plot(cmems_fw_time, cmems_fw_msv, linewidth=0.5, alpha=0.35, color="coral",
            label=f"CMEMS daily (mean={np.nanmean(cmems_fw_msv):.1f} mSv)")

ax.plot(time_array, fw_msv, "-o", markersize=4, linewidth=1.5, color="royalblue",
        label=f"SLCCI monthly (mean={np.nanmean(fw_msv):.1f} mSv)", zorder=5)

ax.axhline(0, color="k", linewidth=0.8)
ax.axhline(np.nanmean(fw_msv), color="royalblue", linestyle="--", linewidth=1, alpha=0.5)
if len(cmems_fw_msv) > 0:
    ax.axhline(np.nanmean(cmems_fw_msv), color="coral", linestyle="--", linewidth=1, alpha=0.5)

ax.set_ylabel("Freshwater Transport (mSv)", fontsize=12)
ax.set_title(
    f"💧 Freshwater Transport — {strait_name} — SLCCI vs CMEMS\n"
    f"S_ref = {S_REF_PSU} PSU | Depth cap: {DEPTH_CAP_M:.0f} m | Salinity: ISAS climatology",
    fontsize=14, fontweight="bold")
ax.legend(fontsize=10)
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.xticks(rotation=45, ha="right")
plt.tight_layout()

if SAVE_OUTPUTS:
    fname = f"09_freshwater_comparison_{satellite_name}_pass{PASS_NUMBER}_{SELECTED_STRAIT}.png"
    plt.savefig(OUTPUT_DIR / fname, dpi=300, bbox_inches="tight")
    print(f"💾 Saved: {fname}")
plt.show()
ds_psal.close()
print("✅ Freshwater transport comparison complete")

In [ ]:
# ════════════════════════════════════════════════════════════════
# 🧪 CELL 13: SALT FLUX — SLCCI vs CMEMS
# ════════════════════════════════════════════════════════════════

# ── SLCCI ──
sf_kgs = salt_flux(
    v_perp=v_perp_matrix, sss=sss_matrix, depth=gate_depth,
    dx=dx_m, depth_cap=DEPTH_CAP_M, rho=RHO)
sf_kts = sf_kgs / 1e6  # kg/s → kt/s

# ── CMEMS (daily, ISAS climatology) ──
if cmems_has_isas:
    cmems_sf_kgs, cmems_sf_time = cmems_salt_flux(
        ds_cmems, depth_cap=DEPTH_CAP_M, rho=RHO_CMEMS, sal_var='psal_isas_surface')
    cmems_sf_kts = cmems_sf_kgs / 1e6
else:
    cmems_sf_kts = np.array([])
    cmems_sf_time = pd.DatetimeIndex([])

print(f"SLCCI SF: mean={np.nanmean(sf_kts):.2f} ± {np.nanstd(sf_kts):.2f} kt/s")
if len(cmems_sf_kts) > 0:
    print(f"CMEMS SF: mean={np.nanmean(cmems_sf_kts):.2f} ± {np.nanstd(cmems_sf_kts):.2f} kt/s")

# ── Plot ──
fig, ax = plt.subplots(figsize=(16, 6))

if len(cmems_sf_kts) > 0:
    ax.plot(cmems_sf_time, cmems_sf_kts, linewidth=0.5, alpha=0.35, color="coral",
            label=f"CMEMS daily (mean={np.nanmean(cmems_sf_kts):.2f} kt/s)")

ax.plot(time_array, sf_kts, "-o", markersize=4, linewidth=1.5, color="darkorange",
        label=f"SLCCI monthly (mean={np.nanmean(sf_kts):.2f} kt/s)", zorder=5)

ax.axhline(0, color="k", linewidth=0.8)
ax.axhline(np.nanmean(sf_kts), color="darkorange", linestyle="--", linewidth=1, alpha=0.5)
if len(cmems_sf_kts) > 0:
    ax.axhline(np.nanmean(cmems_sf_kts), color="coral", linestyle="--", linewidth=1, alpha=0.5)

ax.set_ylabel("Salt Flux (kt/s)", fontsize=12)
ax.set_title(
    f"🧪 Salt Flux — {strait_name} — SLCCI vs CMEMS\n"
    f"ρ_SLCCI={RHO}, ρ_CMEMS={RHO_CMEMS} kg/m³ | Depth cap: {DEPTH_CAP_M:.0f} m | Salinity: ISAS clim",
    fontsize=14, fontweight="bold")
ax.legend(fontsize=10)
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.xticks(rotation=45, ha="right")
plt.tight_layout()

if SAVE_OUTPUTS:
    fname = f"10_salt_flux_comparison_{satellite_name}_pass{PASS_NUMBER}_{SELECTED_STRAIT}.png"
    plt.savefig(OUTPUT_DIR / fname, dpi=300, bbox_inches="tight")
    print(f"💾 Saved: {fname}")
plt.show()
print("✅ Salt flux comparison complete")

In [ ]:
# ════════════════════════════════════════════════════════════════
# 📊 CELL 14: TEMPORAL AGGREGATION — SLCCI vs CMEMS Climatology
# ════════════════════════════════════════════════════════════════

time_idx = pd.DatetimeIndex(time_array)

# ── SLCCI aggregation ──
vt_clim = monthly_climatology(vt_sv, time_idx)
fw_clim = monthly_climatology(fw_msv, time_idx)
vt_annual = annual_mean(vt_sv, time_idx)
fw_annual = annual_mean(fw_msv, time_idx)
vt_rolling = rolling_mean(vt_sv, window=12)

# ── CMEMS aggregation (from daily) ──
cmems_time_idx = pd.DatetimeIndex(cmems_vt_time)
cmems_vt_df = pd.DataFrame({"value": cmems_vt_sv}, index=cmems_time_idx)
cmems_vt_clim = cmems_vt_df.groupby(cmems_vt_df.index.month).agg(mean=("value", "mean"), std=("value", "std")).reset_index()
cmems_vt_clim.columns = ["month", "mean", "std"]
cmems_vt_annual = cmems_vt_df.groupby(cmems_vt_df.index.year).mean()

if len(cmems_fw_msv) > 0:
    cmems_fw_df = pd.DataFrame({"value": cmems_fw_msv}, index=pd.DatetimeIndex(cmems_fw_time))
    cmems_fw_clim = cmems_fw_df.groupby(cmems_fw_df.index.month).agg(mean=("value", "mean"), std=("value", "std")).reset_index()
    cmems_fw_clim.columns = ["month", "mean", "std"]

# ── Plot: 2×2 aggregation overview ──
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
bar_width = 0.35

# (0,0) Monthly climatology: VT
ax = axes[0, 0]
months = vt_clim["month"].values
ax.bar(months - bar_width/2, vt_clim["mean"], bar_width, yerr=vt_clim["std"],
       color="teal", alpha=0.7, capsize=3, edgecolor="k", linewidth=0.5, label="SLCCI")
ax.bar(months + bar_width/2, cmems_vt_clim["mean"], bar_width, yerr=cmems_vt_clim["std"],
       color="coral", alpha=0.7, capsize=3, edgecolor="k", linewidth=0.5, label="CMEMS")
ax.axhline(0, color="k", linewidth=0.8)
ax.set_xticks(range(1, 13))
ax.set_xticklabels(["J","F","M","A","M","J","J","A","S","O","N","D"])
ax.set_ylabel("VT (Sv)", fontsize=12)
ax.set_title("Monthly Climatology — Volume Transport", fontsize=13, fontweight="bold")
ax.legend(fontsize=10)

# (0,1) Monthly climatology: FW
ax = axes[0, 1]
ax.bar(months - bar_width/2, fw_clim["mean"], bar_width, yerr=fw_clim["std"],
       color="royalblue", alpha=0.7, capsize=3, edgecolor="k", linewidth=0.5, label="SLCCI")
if len(cmems_fw_msv) > 0:
    ax.bar(months + bar_width/2, cmems_fw_clim["mean"], bar_width, yerr=cmems_fw_clim["std"],
           color="coral", alpha=0.7, capsize=3, edgecolor="k", linewidth=0.5, label="CMEMS")
ax.axhline(0, color="k", linewidth=0.8)
ax.set_xticks(range(1, 13))
ax.set_xticklabels(["J","F","M","A","M","J","J","A","S","O","N","D"])
ax.set_ylabel("FW (mSv)", fontsize=12)
ax.set_title("Monthly Climatology — Freshwater Transport", fontsize=13, fontweight="bold")
ax.legend(fontsize=10)

# (1,0) Annual mean: VT
ax = axes[1, 0]
slcci_years = vt_annual.index.year
cmems_years = cmems_vt_annual.index
# Find common year range for side-by-side bars
all_years = sorted(set(slcci_years) | set(cmems_years))
x_pos = np.arange(len(all_years))
slcci_vals = [float(vt_annual.loc[vt_annual.index.year == y, "value"].values[0])
              if y in slcci_years else np.nan for y in all_years]
cmems_vals = [float(cmems_vt_annual.loc[y, "value"])
              if y in cmems_years else np.nan for y in all_years]
ax.bar(x_pos - bar_width/2, slcci_vals, bar_width, color="teal", alpha=0.7,
       edgecolor="k", linewidth=0.5, label="SLCCI")
ax.bar(x_pos + bar_width/2, cmems_vals, bar_width, color="coral", alpha=0.7,
       edgecolor="k", linewidth=0.5, label="CMEMS")
ax.axhline(0, color="k", linewidth=0.8)
ax.set_xticks(x_pos[::2])
ax.set_xticklabels([str(all_years[i]) for i in range(0, len(all_years), 2)], rotation=45, ha="right")
ax.set_ylabel("VT (Sv)", fontsize=12)
ax.set_xlabel("Year", fontsize=12)
ax.set_title("Annual Mean — Volume Transport", fontsize=13, fontweight="bold")
ax.legend(fontsize=10)

# (1,1) Time series with rolling means
ax = axes[1, 1]
ax.plot(time_array, vt_sv, alpha=0.25, color="teal", linewidth=0.8, label="SLCCI monthly")
ax.plot(time_array, vt_rolling, color="teal", linewidth=2.5, label="SLCCI 12-mo rolling")

# CMEMS rolling (365-day)
cmems_vt_rolling = cmems_vt_df["value"].rolling(365, center=True, min_periods=180).mean()
ax.plot(cmems_vt_df.index, cmems_vt_rolling, color="coral", linewidth=2.5,
        label="CMEMS 365-day rolling")
ax.plot(cmems_vt_df.index, cmems_vt_df["value"], alpha=0.08, color="coral", linewidth=0.3)

ax.axhline(0, color="k", linewidth=0.8)
ax.set_ylabel("VT (Sv)", fontsize=12)
ax.set_xlabel("Date", fontsize=12)
ax.set_title("Volume Transport — Rolling Means", fontsize=13, fontweight="bold")
ax.legend(fontsize=9)
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha="right")

fig.suptitle(
    f"📊 Temporal Aggregation — {strait_name} — SLCCI vs CMEMS",
    fontsize=16, fontweight="bold", y=1.01)
plt.tight_layout()

if SAVE_OUTPUTS:
    fname = f"11_temporal_aggregation_comparison_{satellite_name}_pass{PASS_NUMBER}_{SELECTED_STRAIT}.png"
    plt.savefig(OUTPUT_DIR / fname, dpi=300, bbox_inches="tight")
    print(f"💾 Saved: {fname}")
plt.show()
print("✅ Temporal aggregation comparison complete")

In [ ]:
# ════════════════════════════════════════════════════════════════
# 📤 CELL 15: EXPORT SUMMARY TABLE
# ════════════════════════════════════════════════════════════════

# ── SLCCI summary ──
summary_slcci = pd.DataFrame({
    "date": time_array,
    "slope_m_100km": slope_series,
    "v_geo_m_s": v_geo,
    "VT_Sv": vt_sv,
    "FW_mSv": fw_msv,
    "SF_kt_s": sf_kts,
})

# ── CMEMS summary (daily) ──
summary_cmems = pd.DataFrame({"date": cmems_vt_time, "CMEMS_VT_Sv": cmems_vt_sv})
summary_cmems["CMEMS_v_perp_m_s"] = cmems_vp_mean_ts
if len(cmems_fw_msv) > 0:
    summary_cmems["CMEMS_FW_mSv"] = cmems_fw_msv
if len(cmems_sf_kts) > 0:
    summary_cmems["CMEMS_SF_kt_s"] = cmems_sf_kts

# ── Print comparison table ──
print("\n" + "=" * 90)
print(f"📤 COMPARISON SUMMARY — {strait_name}")
print("=" * 90)
print(f"\n{'Variable':<25s} {'SLCCI Mean':>12s} {'SLCCI Std':>12s} {'CMEMS Mean':>12s} {'CMEMS Std':>12s}")
print("-" * 85)

# v_geo / v_perp
print(f"{'v_geo / v⊥ (m/s)':<25s} {np.nanmean(v_geo):>12.4f} {np.nanstd(v_geo):>12.4f} "
      f"{np.nanmean(cmems_vp_mean_ts):>12.4f} {np.nanstd(cmems_vp_mean_ts):>12.4f}")
# VT
print(f"{'VT (Sv)':<25s} {np.nanmean(vt_sv):>12.4f} {np.nanstd(vt_sv):>12.4f} "
      f"{np.nanmean(cmems_vt_sv):>12.4f} {np.nanstd(cmems_vt_sv):>12.4f}")
# FW
cmems_fw_mean = np.nanmean(cmems_fw_msv) if len(cmems_fw_msv) > 0 else np.nan
cmems_fw_std = np.nanstd(cmems_fw_msv) if len(cmems_fw_msv) > 0 else np.nan
print(f"{'FW (mSv)':<25s} {np.nanmean(fw_msv):>12.1f} {np.nanstd(fw_msv):>12.1f} "
      f"{cmems_fw_mean:>12.1f} {cmems_fw_std:>12.1f}")
# SF
cmems_sf_mean = np.nanmean(cmems_sf_kts) if len(cmems_sf_kts) > 0 else np.nan
cmems_sf_std = np.nanstd(cmems_sf_kts) if len(cmems_sf_kts) > 0 else np.nan
print(f"{'SF (kt/s)':<25s} {np.nanmean(sf_kts):>12.2f} {np.nanstd(sf_kts):>12.2f} "
      f"{cmems_sf_mean:>12.2f} {cmems_sf_std:>12.2f}")
print("-" * 85)

print(f"\nSLCCI: {len(vt_sv)} monthly values ({time_array[0].strftime('%Y-%m')} → {time_array[-1].strftime('%Y-%m')})")
print(f"CMEMS: {len(cmems_vt_sv)} daily values ({cmems_vt_time[0].strftime('%Y-%m-%d')} → {cmems_vt_time[-1].strftime('%Y-%m-%d')})")

# Save
if SAVE_OUTPUTS:
    csv_slcci = OUTPUT_DIR / f"summary_SLCCI_{satellite_name}_pass{PASS_NUMBER}_{SELECTED_STRAIT}.csv"
    summary_slcci.to_csv(csv_slcci, index=False)
    print(f"\n💾 SLCCI CSV: {csv_slcci}")

    csv_cmems = OUTPUT_DIR / f"summary_CMEMS_{SELECTED_STRAIT}.csv"
    summary_cmems.to_csv(csv_cmems, index=False)
    print(f"💾 CMEMS CSV: {csv_cmems}")

    # Excel with all sheets
    xlsx_path = OUTPUT_DIR / f"comparison_{satellite_name}_pass{PASS_NUMBER}_{SELECTED_STRAIT}.xlsx"
    try:
        with pd.ExcelWriter(xlsx_path) as writer:
            summary_slcci.to_excel(writer, sheet_name="SLCCI_TimeSeries", index=False)
            summary_cmems.to_excel(writer, sheet_name="CMEMS_TimeSeries", index=False)
            vt_clim.to_excel(writer, sheet_name="SLCCI_VT_Climatology", index=False)
            cmems_vt_clim.to_excel(writer, sheet_name="CMEMS_VT_Climatology", index=False)
            fw_clim.to_excel(writer, sheet_name="SLCCI_FW_Climatology", index=False)
            if len(cmems_fw_msv) > 0:
                cmems_fw_clim.to_excel(writer, sheet_name="CMEMS_FW_Climatology", index=False)
        print(f"💾 Excel: {xlsx_path}")
    except ImportError:
        print("⚠️ openpyxl not available — skipping Excel export")

print("\n" + "=" * 90)
print("✅ ALL COMPARATIVE ANALYSIS COMPLETE!")
print("=" * 90)

---## 📋 Comparative Analysis Summary| Step | SLCCI | CMEMS | Module ||------|-------|-------|--------|| Load | Cycle files → DOT | Gate NetCDF (pre-built) | `src.slcci.*` / `utils.load_gate` || Slope | DOT matrix → slope | N/A | `src.slcci.dot` || Profile | Mean DOT vs distance | N/A | `src.slcci.binning` || Map | Pass track on Cartopy | Gate grid overlay | matplotlib + cartopy || Monthly | 12-month DOT profiles | N/A | `src.slcci.binning` || v_geo | −(g/f)×∂η/∂x (uniform) | ugos/vgos → v⊥ (per-point) | `src.physics.geostrophy` / `utils` || VT | Σ v·H·dx (monthly) | Σ v⊥·H·dx (daily) | `src.physics.transport` / `utils` || Salinity | ISAS25 PSAL climatology | ISAS PSAL surface (NetCDF) | xarray || FW | Σ v·(1−S/S_ref)·H·dx | Σ v⊥·(1−S/S_ref)·H·dx | `src.physics.transport` / `utils` || Salt | Σ ρ·(S/1000)·v·H·dx | Σ ρ·(S/1000)·v⊥·H·dx | `src.physics.transport` / `utils` |**Key differences**:- **SLCCI**: single DOT slope → **uniform v_geo** across gate, **monthly** resolution- **CMEMS**: per-point ugos/vgos → **spatially resolved v⊥**, **daily** resolution- **Salinity**: both use **ISAS PSAL climatology** (no interpolation, no invented values)- **Bathymetry**: SLCCI uses GEBCO interpolated to 100 pts; CMEMS uses pre-interpolated GEBCO in NetCDF- **ρ**: SLCCI=1025, CMEMS=1024 kg/m³- **CMEMS data is NOT aggregated** — daily values plotted as-is